# Data Cleaning Project

### Import libraries

In [ ]:
import pandas as pd
import numpy as np

### Data loading and initial inspection

In [ ]:
raw_data = pd.read_csv('dirty_cafe_sales.csv')
df = raw_data.copy()
df.head()

In [ ]:
#find out the number of rows and columns
print(f"Number of rows: {df.shape[0]}")
print(f"Number of columns: {df.shape[1]}")


In [ ]:
#Explore the dataset by columns, non-null counts, and the data types of the columns
df.info()

#### Column Descriptions:

- **Transaction ID**: alpha-numeric text which uniquely identifies each order of products
- **Item**: provides the name of the product being ordered
- **Quantity**: indicates the number of a specific product being ordered
- **Price Per Unit**: provides the individual prices for unique products
- **Total Spent**: provides the amount of money spent on each order
- **Payment Method**: categorical data that describes how products were paid for
- **Location**: categorical data that gives a short description of where the items were consumed
- **Transaction Date**: date that the item was ordered

In [ ]:
#Get summary statistics of the dataset
df.describe()

In [ ]:
#Identify the number of missing values in each column of the dataset
df.isnull().sum()

In [ ]:
#inspect distinct/unique values in each column of the dataset
for i in df.columns:
        print('\n','='* 70,'\n')
        display(df[i].unique())

### Convert all invalid entries in the dataset to null values
### Change the data type of the numerical columns to float in order to perform mathematical calculations


In [ ]:
#Convert all invalid entries('UNKNOWN', 'ERROR') to null
df.replace(['UNKNOWN','ERROR'], np.nan, inplace=True)

#format date to datetime; converts invalid entries to null values
df['Transaction Date'] = pd.to_datetime(df['Transaction Date'], errors='coerce')

#Format the data type of numerical columns to float. 
df[['Quantity','Price Per Unit', 'Total Spent']] = df[['Quantity','Price Per Unit', 'Total Spent']].astype(float)

### Calculate missing values in numerical columns using the formula: Total Spent = Quantity * Price Per Unit


In [ ]:
#Calculate missing rows of numerical date with the formular; Total Spent = Quantity * Price Per Unit
df['Quantity'] = df['Quantity'].fillna(df['Total Spent']/df['Price Per Unit'])
df['Price Per Unit'] = df['Price Per Unit'].fillna(df['Total Spent']/df['Quantity'])
df['Total Spent'] = df['Total Spent'].fillna(df['Quantity'] * df['Price Per Unit'])

### Calculate missing items using the 'Price Per Unit' through mapping

#### Menu Items

The dataset includes the following menu items with their respective price ranges:

|Item|Price($)|
|----|--------|
|Coffee|2|
|Tea|1.5|
|Sandwich|4|
|Salad|5|
|Cake|3|
|Cookie|1|
|Smoothie|4|
|Juice|3|


In [ ]:
#A dictionary does not accept two common keys, therefore, items with the same prices will be filled in differently
#Items with the same prices include Cake and Juice at 3.0, Smoothie and Sandwich at 4.0.
item_prices={1.0:'Cookie',1.5:'Tea',2.0:'Coffee',5.0:'Salad'}

#mapping items to Price Per Unit
mapping = df['Price Per Unit'].map(item_prices)

#Replacing null values in the Item column with the mapped Prices
df['Item'] = df['Item'].fillna(mapping)

In [ ]:
#Items sold at the price of 3.0
items_at_3 = ['Cake','Juice']

#identify missing rows that match the criteria
matching = ((df['Item'].isnull()) & (df['Price Per Unit']==3))

#count the number of rows that match the given criteria
count=matching.sum()

#fill in matching rows in the Item column with random values of the items sold at $3 according to the size of the count
df.loc[matching,'Item'] = np.random.choice(items_at_3, size=count)

In [ ]:
#Using the same method as in Cake and Juice, calculate the rest of the missing values for Sandwich and Smoothie
items_at_4 = ['Sandwich','Smoothie']

#matching rows criteria
rows=((df['Item'].isnull()) & (df['Price Per Unit']==4))

#count of matching rows
row_size = rows.sum()

#calculate missing values
df.loc[rows,'Item'] = np.random.choice(items_at_4, size=row_size)

#### calculate remaining missing numeric data

In [ ]:
#Using 'Item' column to fill the remaining missing numerical values

#Price Per Unit
price_item = {'Cookie':1.0,'Tea':1.5,'Coffee':2.0,'Cake':3.0,'Juice':3.0,'Sandwich':4.0, 'Smoothie':4.0,'Salad':5.0}
df.loc[df['Price Per Unit'].isnull(), 'Price Per Unit'] = df['Item'].map(price_item)

#Quantity
df['Quantity'] = df['Quantity'].fillna(df['Total Spent']/df['Price Per Unit'])

#Total Spent
df['Total Spent'] = df['Total Spent'].fillna(df['Quantity'] * df['Price Per Unit'])

### Fill missing values in categorical columns 'Payment Method' and 'Location' by random sampling


In [ ]:
#Fill Payment Method and Location columns with random sampling; missingness is completely at random

df.loc[df['Payment Method'].isnull(), 'Payment Method'] = df['Payment Method'].dropna().sample(df['Payment Method'].isnull().sum(), replace=True).values
df.loc[df['Location'].isnull(), 'Location'] = df['Location'].dropna().sample(df['Location'].isnull().sum(), replace=True).values

### Interpolate missing date values

In [ ]:
#convert data to integer for interpolation
numeric_date = df['Transaction Date'].astype('int64')

#the code below keeps valid date entries as numeric values and converts invalid entries to null values
numeric_date = numeric_date.where(df['Transaction Date'].notna(), np.nan)

#Interpolate into missing values
df['Transaction Date'] = pd.to_datetime(numeric_date.interpolate())

### Feature Engineering

In [ ]:
#convert date to string
df['Transaction Date'] = df['Transaction Date'].astype(str)

#Create a temporal field that seperates the date(e.g 2023-12-31) from the time (e.g 00:00:00)
df['Date'] = df['Transaction Date'].str.split(' ').str[0]

#create Day, Month, Year features from the temporal field
df['Day'] = df['Date'].str.split('-').str[2].astype(int)
df['Month'] = df['Date'].str.split('-').str[1].astype(int)
df['Year'] = df['Date'].str.split('-').str[0].astype(int)


In [ ]:
#the other date fields will no longer be needed, therefore, they are dropped
df.drop(['Transaction Date','Date'], axis=1, inplace=True)

### Find and drop rows with missing and duplicated values

In [ ]:
df[df['Quantity'].isnull()]

In [ ]:
df[df['Quantity'].isnull()]

In [ ]:
df[df['Total Spent'].isnull()]

In [ ]:

df.dropna(inplace=True)

In [ ]:
df.duplicated().sum()
#No duplicates found

### Final Data Inspection

In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
#compare the number of rows and columns in the dirty and cleaned datasets
print('Raw Data:',raw_data.shape)
print('Cleaned data:',df.shape)

In [ ]:
#Initial null counts (df.isnull().sum())
raw_data.isnull().sum()

In [ ]:
#Final null counts
df.isnull().sum()

In [ ]:
df.info()

### Save cleaned dataset as a csv file

In [ ]:
df.to_csv('cleaned_cafe_sales.csv', index=False)

#### Cleaning summary

* Handled missing data
* Dropped 26 rows of remaining missing data
* No duplicated rows were found
* Performed feature engineering on the date column for better analysis
* Saved the cleaned dataset as a csv file